# FrustraMPNN: Ultra-Fast Frustration Prediction

<img src="https://img.shields.io/badge/License-MIT-blue.svg" alt="License: MIT">
<img src="https://img.shields.io/badge/Python-3.10+-green.svg" alt="Python 3.10+">

**FrustraMPNN** is a message-passing neural network for ultra-fast prediction of single-residue local energetic frustration profiles in proteins.

## What is Frustration?

Local energetic frustration quantifies how favorable or unfavorable a residue's local environment is compared to alternative amino acids. This concept from energy landscape theory helps identify:

- **Highly frustrated residues** (red): Unfavorable local interactions, often at functional sites
- **Minimally frustrated residues** (green): Favorable interactions, typically in the protein core
- **Neutral residues** (gray): Average interactions

## This Notebook

This notebook provides zero-installation access to FrustraMPNN predictions:

1. **Install** - One-click installation of FrustraMPNN
2. **Upload/Fetch PDB** - Upload your structure or fetch from RCSB
3. **Predict** - Run frustration prediction
4. **Visualize** - Interactive plots and 3D structure viewer
5. **Validate** - Compare with physics-based frustrapy (optional)
6. **Download** - Export results as CSV

**Speed**: FrustraMPNN is 1,000-4,500x faster than physics-based methods!

## 1. Installation

Run this cell to install FrustraMPNN and its dependencies. This takes about 2-3 minutes.

In [ ]:
# @title Install FrustraMPNN { display-mode: "form" }
# @markdown Click the play button to install FrustraMPNN.
# @markdown This installs the package with visualization support.

import subprocess
import sys


def install_frustrampnn():
    """Install FrustraMPNN from GitHub."""
    print("Installing FrustraMPNN...")
    print("=" * 50)

    # Install from GitHub with viz extras
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "frustrampnn[viz,esm] @ git+https://github.com/schoederlab/frustraMPNN.git@package-and-colab",
        ]
    )

    print("\nInstallation complete!")
    print("=" * 50)

    # Verify installation
    try:
        import frustrampnn

        print(f"FrustraMPNN version: {frustrampnn.__version__}")

        import torch

        print(f"PyTorch version: {torch.__version__}")
        print(f"CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"   GPU: {torch.cuda.get_device_name(0)}")
    except ImportError as e:
        print(f"Import error: {e}")
        raise


install_frustrampnn()

## 2. Load Your Protein Structure

You can either:
- **Upload a PDB file** from your computer
- **Fetch from RCSB** using a PDB ID (e.g., "1UBQ")

In [ ]:
# @title Load PDB Structure { display-mode: "form" }
# @markdown Choose how to load your protein structure:

import os

# @markdown ---
# @markdown ### Option 1: Fetch from RCSB
pdb_id = "1UBQ"  # @param {type:"string"}
# @markdown Enter a 4-letter PDB ID (e.g., "1UBQ", "4BLM", "1NLU")

# @markdown ---
# @markdown ### Option 2: Upload a file
upload_file = False  # @param {type:"boolean"}
# @markdown Check this box to upload a PDB file from your computer

# @markdown ---
chain_id = "A"  # @param {type:"string"}
# @markdown Chain ID to analyze (leave empty for all chains)


def fetch_pdb(pdb_id: str) -> str:
    """Fetch PDB file from RCSB."""
    import urllib.request

    pdb_id = pdb_id.upper().strip()
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    output_path = f"{pdb_id}.pdb"

    print(f"Fetching {pdb_id} from RCSB...")
    urllib.request.urlretrieve(url, output_path)
    print(f"Downloaded: {output_path}")

    return output_path


def upload_pdb() -> str:
    """Upload PDB file from user's computer."""
    from google.colab import files

    print("Please select a PDB file to upload...")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file uploaded!")

    filename = list(uploaded.keys())[0]
    print(f"Uploaded: {filename}")

    return filename


# Load the structure
if upload_file:
    pdb_path = upload_pdb()
else:
    pdb_path = fetch_pdb(pdb_id)

# Parse and display info
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", pdb_path)

chains = [c.id for c in structure.get_chains()]
residues = sum(1 for r in structure.get_residues() if r.id[0] == " ")

print("\nStructure Info:")
print(f"   File: {pdb_path}")
print(f"   Chains: {', '.join(chains)}")
print(f"   Residues: {residues}")

# Store for later use
PROTEIN_PATH = pdb_path
CHAIN_ID = chain_id if chain_id.strip() else None
print(f"\nWill analyze: Chain {CHAIN_ID if CHAIN_ID else 'all chains'}")

## 3. Load Model and Run Prediction

Load the pretrained FrustraMPNN model and predict frustration values for all residues.

In [ ]:
# @title Download Model Checkpoint { display-mode: "form" }
# @markdown Download the pretrained FrustraMPNN model checkpoint.

import urllib.request

# Model checkpoint URL (hosted on GitHub releases or similar)
CHECKPOINT_URL = "https://github.com/schoederlab/frustraMPNN/releases/download/v1.0.0/frustrampnn_fireprot_balanced.ckpt"
CHECKPOINT_PATH = "frustrampnn_fireprot_balanced.ckpt"

# Also need vanilla model weights
VANILLA_WEIGHTS_URL = "https://github.com/schoederlab/frustraMPNN/releases/download/v1.0.0/vanilla_model_weights.tar.gz"


def download_checkpoint():
    """Download model checkpoint if not present."""
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Checkpoint already exists: {CHECKPOINT_PATH}")
        return

    print("Downloading model checkpoint...")
    print(f"   URL: {CHECKPOINT_URL}")

    try:
        urllib.request.urlretrieve(CHECKPOINT_URL, CHECKPOINT_PATH)
        print(f"Downloaded: {CHECKPOINT_PATH}")
    except Exception as e:
        print(f"Could not download from URL: {e}")
        print("\nManual download instructions:")
        print("   1. Download the checkpoint from the GitHub releases page")
        print("   2. Upload it to this Colab session")
        print("   3. Update CHECKPOINT_PATH variable above")
        raise


def download_vanilla_weights():
    """Download vanilla ProteinMPNN weights if not present."""
    weights_dir = "vanilla_model_weights"
    if os.path.exists(weights_dir):
        print(f"Vanilla weights already exist: {weights_dir}/")
        return

    print("Downloading vanilla ProteinMPNN weights...")

    try:
        import tarfile

        tar_path = "vanilla_model_weights.tar.gz"
        urllib.request.urlretrieve(VANILLA_WEIGHTS_URL, tar_path)

        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(".")

        os.remove(tar_path)
        print(f"Extracted: {weights_dir}/")
    except Exception as e:
        print(f"Could not download vanilla weights: {e}")
        print("   The model may still work if weights are embedded in checkpoint.")


# Download files
download_checkpoint()
download_vanilla_weights()

In [ ]:
# @title Load Model { display-mode: "form" }
# @markdown Load the pretrained FrustraMPNN model.

import torch

from frustrampnn import FrustraMPNN

print("Loading FrustraMPNN model...")
print(f"   Checkpoint: {CHECKPOINT_PATH}")
print(f"   Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

# Load model
model = FrustraMPNN.from_pretrained(CHECKPOINT_PATH)

print("\nModel loaded successfully!")
print(f"   {model}")

In [ ]:
# @title Run Frustration Prediction { display-mode: "form" }
# @markdown Predict frustration values for all residues.
# @markdown This generates 20 predictions per residue (one for each amino acid variant).

import time

print("Running frustration prediction...")
print(f"   PDB: {PROTEIN_PATH}")
print(f"   Chain: {CHAIN_ID if CHAIN_ID else 'all'}")
print()

# Run prediction
start_time = time.time()

chains_to_analyze = [CHAIN_ID] if CHAIN_ID else None
results = model.predict(PROTEIN_PATH, chains=chains_to_analyze, show_progress=True)

elapsed = time.time() - start_time

# Summary
n_positions = results["position"].nunique()
n_predictions = len(results)

print("\nPrediction complete!")
print(f"   Positions analyzed: {n_positions}")
print(f"   Total predictions: {n_predictions} ({n_positions} × 20 amino acids)")
print(f"   Time elapsed: {elapsed:.2f} seconds")
print(f"   Speed: {n_predictions / elapsed:.1f} predictions/second")

# Show sample
print("\nSample results:")
print(results.head(10).to_string(index=False))

## 4. Visualize Results

### 4.1 Single-Residue Frustration Plot

Visualize frustration values for all 20 amino acid variants at a specific position.

In [ ]:
# @title Single-Residue Frustration Plot { display-mode: "form" }
# @markdown Select a position to visualize frustration for all amino acid variants.

from frustrampnn.visualization import plot_single_residue_plotly

# @markdown ---
position_to_plot = 0  # @param {type:"integer"}
# @markdown Position to plot (0-indexed). For example, position 0 is the first residue.

# @markdown ---
# @markdown **Note**: The plot shows frustration values for all 20 amino acids at this position.
# @markdown - **Red bars**: Highly frustrated (≤ -1.0)
# @markdown - **Gray bars**: Neutral (-1.0 to 0.58)
# @markdown - **Green bars**: Minimally frustrated (≥ 0.58)
# @markdown - **Blue bar**: Native (wild-type) residue

# Get available positions
available_positions = sorted(results["position"].unique())
print(f"Available positions: {min(available_positions)} to {max(available_positions)}")

if position_to_plot not in available_positions:
    print(f"Position {position_to_plot} not in results. Using position {available_positions[0]}.")
    position_to_plot = available_positions[0]

# Create interactive plot
chain_for_plot = CHAIN_ID if CHAIN_ID else results["chain"].iloc[0]
fig = plot_single_residue_plotly(
    results,
    position=position_to_plot,
    chain=chain_for_plot,
    width=900,
    height=500,
)

fig.show()

### 4.2 Frustration Heatmap

Visualize frustration values for all positions and all amino acid variants as a heatmap.

In [ ]:
# @title Frustration Heatmap { display-mode: "form" }
# @markdown Interactive heatmap showing frustration across all positions.

from frustrampnn.visualization import plot_frustration_heatmap_plotly

# Create heatmap
chain_for_heatmap = CHAIN_ID if CHAIN_ID else results["chain"].iloc[0]
fig = plot_frustration_heatmap_plotly(
    results,
    chain=chain_for_heatmap,
    width=1200,
    height=600,
    colorscale="RdYlGn_r",  # Red (frustrated) to Green (minimally frustrated)
    zmin=-3,
    zmax=3,
)

fig.show()

print("\nHeatmap interpretation:")
print("   - Red: Highly frustrated (unfavorable)")
print("   - Yellow: Neutral")
print("   - Green: Minimally frustrated (favorable)")
print("   - X-axis: Residue position (1-indexed)")
print("   - Y-axis: Amino acid variant")

### 4.3 3D Structure Viewer

Visualize the protein structure colored by frustration.

In [ ]:
# @title 3D Structure Viewer { display-mode: "form" }
# @markdown Interactive 3D visualization of the protein colored by average frustration.

import py3Dmol

# @markdown ---
color_by = "average"  # @param ["average", "wildtype", "most_frustrated"]
# @markdown - **average**: Mean frustration across all variants
# @markdown - **wildtype**: Frustration of the native residue
# @markdown - **most_frustrated**: Most frustrated variant at each position


def get_frustration_color(value: float) -> str:
    """Convert frustration value to RGB color."""
    # Normalize to 0-1 range (assuming -3 to 3 range)
    normalized = (value + 3) / 6
    normalized = max(0, min(1, normalized))

    # Red (frustrated) to Green (minimally frustrated)
    if normalized < 0.5:
        # Red to Yellow
        r = 255
        g = int(255 * (normalized * 2))
        b = 0
    else:
        # Yellow to Green
        r = int(255 * (1 - (normalized - 0.5) * 2))
        g = 255
        b = 0

    return f"rgb({r},{g},{b})"


def create_3d_viewer(pdb_path: str, results_df, chain: str, color_mode: str):
    """Create py3Dmol viewer colored by frustration."""
    # Read PDB content
    with open(pdb_path) as f:
        pdb_content = f.read()

    # Calculate per-position frustration based on mode
    chain_results = results_df[results_df["chain"] == chain] if chain else results_df

    if color_mode == "average":
        pos_frustration = chain_results.groupby("position")["frustration_pred"].mean()
    elif color_mode == "wildtype":
        # Get wildtype frustration (where mutation == wildtype)
        wt_mask = chain_results["mutation"] == chain_results["wildtype"]
        pos_frustration = chain_results[wt_mask].set_index("position")["frustration_pred"]
    else:  # most_frustrated
        pos_frustration = chain_results.groupby("position")["frustration_pred"].min()

    # Create viewer
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_content, "pdb")

    # Color by frustration
    for pos, frust in pos_frustration.items():
        color = get_frustration_color(frust)
        # Position is 0-indexed, PDB residue numbers are typically 1-indexed
        view.setStyle(
            {"resi": pos + 1, "chain": chain} if chain else {"resi": pos + 1},
            {"cartoon": {"color": color}},
        )

    view.zoomTo()
    view.setBackgroundColor("white")

    return view


# Create and display viewer
chain_for_3d = CHAIN_ID if CHAIN_ID else results["chain"].iloc[0]
viewer = create_3d_viewer(PROTEIN_PATH, results, chain_for_3d, color_by)
viewer.show()

print(f"\nColoring mode: {color_by}")
print("   Red → Yellow → Green = Frustrated → Neutral → Minimally frustrated")
print("\nControls:")
print("   - Left-click + drag: Rotate")
print("   - Right-click + drag: Translate")
print("   - Scroll: Zoom")

## 5. Validate Against frustrapy (Optional)

Compare FrustraMPNN predictions with physics-based frustrapy calculations.

**Note**: frustrapy is much slower than FrustraMPNN. Only validate a few positions at a time.

In [ ]:
# @title Validate Against frustrapy { display-mode: "form" }
# @markdown Compare FrustraMPNN predictions with physics-based frustrapy.

# @markdown ---
run_validation = False  # @param {type:"boolean"}
# @markdown Check this box to run validation. This can take several minutes!

# @markdown ---
positions_to_validate = "0,1,2"  # @param {type:"string"}
# @markdown Comma-separated list of positions to validate (0-indexed).
# @markdown Keep this small (3-5 positions) as frustrapy is slow.

if run_validation:
    import time

    from frustrampnn.validation import compare_with_frustrapy

    # Parse positions
    positions = [int(p.strip()) for p in positions_to_validate.split(",")]
    chain_for_validation = CHAIN_ID if CHAIN_ID else results["chain"].iloc[0]

    print("Running frustrapy validation...")
    print(f"   Positions: {positions}")
    print(f"   Chain: {chain_for_validation}")
    print("\nThis may take several minutes...")
    print()

    start_time = time.time()

    try:
        comparison = compare_with_frustrapy(
            pdb_path=PROTEIN_PATH,
            chain=chain_for_validation,
            frustrampnn_results=results,
            positions=positions,
        )

        elapsed = time.time() - start_time

        print(f"\nValidation complete! ({elapsed:.1f} seconds)")
        print("\nCorrelation Metrics:")
        print(f"   Spearman correlation: {comparison.spearman:.3f}")
        print(f"   Pearson correlation:  {comparison.pearson:.3f}")
        print(f"   RMSE:                 {comparison.rmse:.3f}")
        print(f"   MAE:                  {comparison.mae:.3f}")
        print(f"\nPositions compared: {comparison.n_positions}")
        print(f"   Data points: {comparison.n_datapoints}")

        # Show comparison plot
        print("\nGenerating comparison plot...")
        fig = comparison.plot()
        fig.show()

    except Exception as e:
        print(f"Validation failed: {e}")
        print("\nTips:")
        print("   - Make sure frustrapy is installed")
        print("   - Check that the PDB file is valid")
        print("   - Try fewer positions")

else:
    print("Validation skipped. Check 'run_validation' to enable.")
    print("   Note: frustrapy validation is slow (~2-3 min per position)")

## 6. Download Results

Export your results as a CSV file for further analysis.

In [ ]:
# @title Download Results { display-mode: "form" }
# @markdown Save and download the frustration predictions.

from google.colab import files

# Generate output filename
pdb_name = os.path.splitext(os.path.basename(PROTEIN_PATH))[0]
output_filename = f"{pdb_name}_frustration_predictions.csv"

# Add frustration classification
from frustrampnn.visualization import classify_frustration

results_export = results.copy()
results_export["frustration_class"] = results_export["frustration_pred"].apply(classify_frustration)

# Add 1-indexed position for convenience
results_export["position_1indexed"] = results_export["position"] + 1

# Reorder columns for clarity
column_order = [
    "pdb",
    "chain",
    "position",
    "position_1indexed",
    "wildtype",
    "mutation",
    "frustration_pred",
    "frustration_class",
]
results_export = results_export[column_order]

# Save to CSV
results_export.to_csv(output_filename, index=False)

print(f"Results saved to: {output_filename}")
print(f"   Rows: {len(results_export)}")
print(f"   Columns: {', '.join(results_export.columns)}")

# Summary statistics
print("\nSummary Statistics:")
print(f"   Mean frustration: {results_export['frustration_pred'].mean():.3f}")
print(f"   Std frustration:  {results_export['frustration_pred'].std():.3f}")
print(f"   Min frustration:  {results_export['frustration_pred'].min():.3f}")
print(f"   Max frustration:  {results_export['frustration_pred'].max():.3f}")

# Class distribution
class_counts = results_export["frustration_class"].value_counts()
print("\nFrustration Class Distribution:")
for cls, count in class_counts.items():
    pct = 100 * count / len(results_export)
    print(f"   {cls}: {count} ({pct:.1f}%)")

# Download
print(f"\nDownloading {output_filename}...")
files.download(output_filename)

## 7. Advanced: Batch Processing

Process multiple PDB files at once.

In [ ]:
# @title Batch Processing { display-mode: "form" }
# @markdown Upload multiple PDB files and process them all at once.

run_batch = False  # @param {type:"boolean"}
# @markdown Check this box to enable batch processing.

if run_batch:
    from google.colab import files

    print("Please upload your PDB files...")
    uploaded = files.upload()

    if uploaded:
        pdb_files = list(uploaded.keys())
        print(f"\nUploaded {len(pdb_files)} files:")
        for f in pdb_files:
            print(f"   - {f}")

        print(f"\nProcessing {len(pdb_files)} structures...")

        # Run batch prediction
        batch_results = model.predict_batch(
            pdb_files,
            chains=None,  # All chains
            show_progress=True,
        )

        print("\nBatch processing complete!")
        print(f"   Total predictions: {len(batch_results)}")

        # Save combined results
        batch_output = "batch_frustration_predictions.csv"
        batch_results.to_csv(batch_output, index=False)
        print(f"\nSaved to: {batch_output}")

        files.download(batch_output)
    else:
        print("No files uploaded.")
else:
    print("Batch processing disabled. Check 'run_batch' to enable.")

---

## Citation

If you use FrustraMPNN in your research, please cite:

```bibtex
@article{beining2024frustrampnn,
  title={FrustraMPNN: Ultra-fast deep learning prediction of single-residue local energetic frustration},
  author={Beining*, Max and Engelberger*, Felipe and Schoeder, Clara T. and Ram{\'\i}rez-Sarmiento, C{\'e}sar A. and Meiler, Jens},
  journal={bioRxiv},
  year={2024},
  doi={10.1101/2024.XX.XX.XXXXXX},
  note={*These authors contributed equally}
}
```

## Resources

- **GitHub**: [github.com/schoederlab/frustraMPNN](https://github.com/schoederlab/frustraMPNN)
- **Documentation**: [frustrampnn.readthedocs.io](https://frustrampnn.readthedocs.io)
- **frustrapy**: [github.com/engelberger/frustrapy](https://github.com/engelberger/frustrapy)
- **FrustratometeR**: [frustratometer.qb.fcen.uba.ar](http://frustratometer.qb.fcen.uba.ar)

## License

MIT License - see [LICENSE](https://github.com/schoederlab/frustraMPNN/blob/main/LICENSE) for details.